# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FARNHELL/ML-INTERNSHIP/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1 — freshness and refresh timing.** The assigned paper reports that 365+ day content refreshed within 30 days had a 3.2x health-score difference and 57x more impressions than older stale content. That is a useful observed comparison, and the paper itself labels the study observational. **Methodology question:** how were pages assigned to the refreshed and stale groups, and were they comparable on prior visibility, topic, client, and original quality before the update? Without a matched or before/after design, the gap may partly reflect which pages were selected for refresh rather than the refresh itself.

**Finding 2 — exploratory growth classification.** The paper reports 71% holdout accuracy for a logistic regression separating growing from declining pages. This is a useful exploratory result, especially because the appendix calls it descriptive. **Methodology question:** was the 80/20 holdout split grouped by brand or ordered in time, and what was the class base rate and precision@K? A random row holdout can place pages from the same brand on both sides, and accuracy alone does not show whether the model improves a ranked editorial queue.

The code below records the page-level source evidence used for these questions. The questions target label and validation design, not the authors or the value of the descriptive findings.

In [1]:
from pathlib import Path
from pypdf import PdfReader
import pandas as pd

ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'docs' / 'flyrank-seo-research-march-2026.pdf').exists():
        ROOT = candidate
        break
paper_path = ROOT / 'docs' / 'flyrank-seo-research-march-2026.pdf'
reader = PdfReader(paper_path)
assert len(reader.pages) == 36
paper_evidence = pd.DataFrame([
    {
        'paper_page': 9,
        'reported_finding': '365+ day pages refreshed within 30 days: 3.2x health and 57x impressions versus older stale pages.',
        'methodology_question': 'Were refreshed and stale pages comparable before refresh, or was refresh selection confounded by prior value?',
    },
    {
        'paper_page': 29,
        'reported_finding': 'Exploratory logistic regression: 71% holdout accuracy for growing versus declining pages.',
        'methodology_question': 'Was the 80/20 holdout grouped by brand or ordered in time, and how does ranking precision compare with the base rate?',
    },
])
print('Assigned paper:', paper_path.name, '| pages:', len(reader.pages))
print(paper_evidence.to_string(index=False))
# Confirm that the cited pages contain the key wording used above.
for page_number, phrase in [(9, '3.2x health boost'), (29, '71% holdout accuracy')]:
    page_text = reader.pages[page_number - 1].extract_text()
    print('Page {} contains {!r}: {}'.format(page_number, phrase, phrase in page_text))

Assigned paper: flyrank-seo-research-march-2026.pdf | pages: 36
 paper_page                                                                                   reported_finding                                                                                                  methodology_question
          9 365+ day pages refreshed within 30 days: 3.2x health and 57x impressions versus older stale pages.         Were refreshed and stale pages comparable before refresh, or was refresh selection confounded by prior value?
         29          Exploratory logistic regression: 71% holdout accuracy for growing versus declining pages. Was the 80/20 holdout grouped by brand or ordered in time, and how does ranking precision compare with the base rate?
Page 9 contains '3.2x health boost': True
Page 29 contains '71% holdout accuracy': True


## 2. My model under an honest split (before/after)

The Week 5 model already uses a grouped split by `client_hash_id`. To show why that matters, I run the same Logistic Regression twice on the same March-feature / April-label frame: **before** with a naive random row split and **after** with the Week 5 grouped client split. Both use the same five allowed features, fixed `random_state=42`, the same target, the same 75/25 test fraction, precision@K, and ROC-AUC.

The random split is not a deployment claim. It is a leakage stress test: it allows the same client to appear in train and test, whereas the grouped split asks whether the model transfers to unseen clients. The code prints client overlap, base rate, and every metric for both states.

In [2]:
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

OUTPUT_ROOT = ROOT / 'work' / 'outputs'
FEATURES = ['feat_impr_7d', 'feat_clicks_7d', 'feat_pos_7d', 'feat_scroll_7d', 'content_type']
NUMERIC_FEATURES = FEATURES[:4]
CATEGORICAL_FEATURES = ['content_type']
KS = [10, 20, 50, 100, 500]


def load_hf_token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN').strip()
    except (ImportError, KeyError, AttributeError):
        try:
            from dotenv import load_dotenv
            load_dotenv(ROOT / '.env')
        except ImportError:
            pass
        return os.environ.get('HF_TOKEN', '').strip()


def warehouse_file(filename, token):
    from huggingface_hub import hf_hub_download
    try:
        return hf_hub_download('FlyRank/internship-warehouse', filename=filename, repo_type='dataset', token=token)
    except PermissionError:
        return hf_hub_download('FlyRank/internship-warehouse', filename=filename, repo_type='dataset', token=token, local_dir=Path(tempfile.gettempdir()) / 'flyrank_w06_warehouse')


def load_model_frame():
    token = load_hf_token()
    if not token:
        raise RuntimeError('HF_TOKEN is not configured')
    daily_columns = ['report_date', 'client_hash_id', 'content_hash_id', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'scroll_events']
    march = pd.read_parquet(warehouse_file('fact_content_daily_performance/month=2026-03/data_0.parquet', token), columns=daily_columns)
    april = pd.read_parquet(warehouse_file('fact_content_daily_performance/month=2026-04/data_0.parquet', token), columns=['client_hash_id', 'content_hash_id', 'gsc_data_available', 'gsc_impressions'])
    content = pd.read_parquet(warehouse_file('dim_content.parquet', token), columns=['client_hash_id', 'content_hash_id', 'content_updated_date', 'content_type'])
    march['report_date'] = pd.to_datetime(march['report_date'])
    march = march[march['gsc_data_available'].astype(bool)].copy()
    april = april[april['gsc_data_available'].astype(bool)].copy()
    keys = ['client_hash_id', 'content_hash_id']
    march_total = march.groupby(keys, as_index=False).agg(march_impressions_30d=('gsc_impressions', 'sum'))
    feature_frame = march[march['report_date'].ge('2026-03-25')].groupby(keys, as_index=False).agg(
        feat_impr_7d=('gsc_impressions', 'mean'),
        feat_clicks_7d=('gsc_clicks', 'mean'),
        feat_pos_7d=('gsc_avg_position', 'mean'),
        feat_scroll_7d=('scroll_events', 'mean'),
        ga4_available_days=('ga4_data_available', 'sum'),
    )
    april_total = april.groupby(keys, as_index=False).agg(april_impressions_30d=('gsc_impressions', 'sum'))
    content['content_updated_date'] = pd.to_datetime(content['content_updated_date'], errors='coerce')
    frame = march_total.merge(feature_frame, on=keys).merge(april_total, on=keys).merge(content, on=keys, how='left')
    frame['days_since_last_update'] = (pd.Timestamp('2026-03-31') - frame['content_updated_date']).dt.days
    # Same eligible cohort as Week 4/5. This field is cohort construction, never a model feature.
    frame = frame[frame['days_since_last_update'].ge(0)].copy()
    frame['observed_forward_decline'] = (frame['march_impressions_30d'].gt(0) & frame['april_impressions_30d'].lt(.8 * frame['march_impressions_30d'])).astype(int)
    frame['data_source'] = 'warehouse: March features, April held-out label'
    return frame


try:
    audit_frame = load_model_frame()
    print('Using warehouse data for the validation audit.')
except Exception as warehouse_error:
    starter = pd.read_csv(ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv')
    audit_frame = starter.assign(
        feat_impr_7d=starter['impressions_90d'] / 90,
        feat_clicks_7d=starter['clicks_90d'] / 90,
        feat_pos_7d=starter['avg_position'],
        feat_scroll_7d=starter['scroll_events_90d'] / 90,
        ga4_available_days=np.nan,
        observed_forward_decline=starter['trend_direction'].eq('down').astype(int),
        data_source='starter CSV fallback',
    )
    print('Warehouse unavailable; starter fallback used: {}'.format(type(warehouse_error).__name__))


def make_model():
    preprocess = ColumnTransformer([
        ('numeric', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), NUMERIC_FEATURES),
        ('content_type', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAL_FEATURES),
    ])
    return Pipeline([('preprocess', preprocess), ('logistic_regression', LogisticRegression(max_iter=1000, random_state=42))])


def evaluate(train_idx, test_idx, split_name):
    train = audit_frame.iloc[train_idx]
    test = audit_frame.iloc[test_idx]
    fitted = make_model().fit(train[FEATURES], train['observed_forward_decline'])
    score = fitted.predict_proba(test[FEATURES])[:, 1]
    ranked = test.assign(model_score=score).sort_values('model_score', ascending=False).reset_index(drop=True)
    return {
        'split': split_name,
        'train_rows': len(train),
        'test_rows': len(test),
        'train_clients': train['client_hash_id'].nunique(),
        'test_clients': test['client_hash_id'].nunique(),
        'client_overlap': len(set(train['client_hash_id']) & set(test['client_hash_id'])),
        'test_base_rate': ranked['observed_forward_decline'].mean(),
        **{'precision@{}'.format(k): ranked.head(k)['observed_forward_decline'].mean() for k in KS},
        'ROC-AUC': roc_auc_score(ranked['observed_forward_decline'], ranked['model_score']),
    }, ranked

random_train, random_test = train_test_split(np.arange(len(audit_frame)), test_size=.25, random_state=42, stratify=audit_frame['observed_forward_decline'])
group_train, group_test = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=42).split(audit_frame, groups=audit_frame['client_hash_id']))
before_result, random_ranked = evaluate(random_train, random_test, 'Before: random row split')
after_result, grouped_ranked = evaluate(group_train, group_test, 'After: grouped client split')
validation_comparison = pd.DataFrame([before_result, after_result])
print('Eligible rows: {:,}; overall base rate: {:.3f}'.format(len(audit_frame), audit_frame['observed_forward_decline'].mean()))
print(validation_comparison.to_string(index=False, formatters={column: '{:.3f}'.format for column in validation_comparison.columns if column.startswith('precision@') or column in ['test_base_rate', 'ROC-AUC']}))

Using warehouse data for the validation audit.
Eligible rows: 23,936; overall base rate: 0.571
                      split  train_rows  test_rows  train_clients  test_clients  client_overlap test_base_rate precision@10 precision@20 precision@50 precision@100 precision@500 ROC-AUC
   Before: random row split       17952       5984             31            31              30          0.571        0.800        0.700        0.800         0.780         0.606   0.558
After: grouped client split       21887       2049             24             8               0          0.694        0.600        0.700        0.680         0.660         0.654   0.463


## 3. Leakage audit

The five model inputs are audited one by one against the April label window and client identity. The model uses March 25-31 trailing daily values for the four numeric features; the label is defined only from April totals compared with March totals. `content_type` is static metadata. `client_hash_id` is used only to create the grouped split and never reaches the feature pipeline.

- `feat_impr_7d`: March GSC impressions, strictly before April. It does not contain the April label window or client ID, but it is immediately before the prediction boundary and therefore a recent-state signal, not a causal explanation.
- `feat_clicks_7d`: March GSC clicks, strictly before April. It is not label-derived and excludes client ID; like impressions, it is a close-to-boundary operational signal.
- `feat_pos_7d`: March GSC average position, strictly before April. It does not overlap the label, but a value of zero can mean no position data, so the number is measurement-ambiguous rather than automatically clean evidence of poor rank.
- `feat_scroll_7d`: March GA4 scroll events, strictly before April. It does not use April data or client ID, but it is **borderline** because GA4-unavailable rows can be zero-filled; the audit prints how many items have no GA4-available day. That is a data-availability limitation, not future leakage.
- `content_type`: static content metadata, so it is available before April and contains no client ID. It can still carry client or publishing-practice differences indirectly; the grouped split is the protection against learning a client-specific category pattern.
- `observed_forward_decline`: the label only. It uses April impressions relative to March and never enters the model feature list.

The audit table and checks below make the timing and exclusions inspectable rather than assumed.

In [3]:
excluded_columns = ['trend_direction', 'trend_pct', 'gsc_data_available', 'ga4_data_available', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai']
leakage_audit = pd.DataFrame([
    {'field': 'feat_impr_7d', 'availability': 'March 25-31 GSC impressions', 'April label overlap': 'No', 'client ID used as feature': 'No', 'audit result': 'Allowed; near-boundary signal'},
    {'field': 'feat_clicks_7d', 'availability': 'March 25-31 GSC clicks', 'April label overlap': 'No', 'client ID used as feature': 'No', 'audit result': 'Allowed; near-boundary signal'},
    {'field': 'feat_pos_7d', 'availability': 'March 25-31 GSC average position', 'April label overlap': 'No', 'client ID used as feature': 'No', 'audit result': 'Allowed; zero can mean no position data'},
    {'field': 'feat_scroll_7d', 'availability': 'March 25-31 GA4 scroll events', 'April label overlap': 'No', 'client ID used as feature': 'No', 'audit result': 'Borderline: GA4 zero-fill can mean unavailable'},
    {'field': 'content_type', 'availability': 'Static metadata', 'April label overlap': 'No', 'client ID used as feature': 'No', 'audit result': 'Allowed; grouped split guards client mix'},
    {'field': 'observed_forward_decline', 'availability': 'April totals versus March totals', 'April label overlap': 'This is the label', 'client ID used as feature': 'No', 'audit result': 'Never a feature'},
])
print(leakage_audit.to_string(index=False))
print('\nFeature list used by model:', FEATURES)
print('Excluded from feature list:', excluded_columns)
print('Any April total in model features:', any('april' in feature.lower() for feature in FEATURES))
print('Any client identifier in model features:', any('client' in feature.lower() for feature in FEATURES))
print('Grouped-split client overlap:', after_result['client_overlap'])
print('feat_pos_7d equal to zero: {:,} of {:,}'.format(int(audit_frame['feat_pos_7d'].eq(0).sum()), len(audit_frame)))
if audit_frame['ga4_available_days'].notna().any():
    print('Items with zero GA4-available days in the 7-day feature window: {:,} of {:,}'.format(int(audit_frame['ga4_available_days'].eq(0).sum()), len(audit_frame)))
else:
    print('GA4 availability is unavailable in the starter fallback; scroll is therefore not interpretable there.')

                   field                     availability April label overlap client ID used as feature                                   audit result
            feat_impr_7d      March 25-31 GSC impressions                  No                        No                  Allowed; near-boundary signal
          feat_clicks_7d           March 25-31 GSC clicks                  No                        No                  Allowed; near-boundary signal
             feat_pos_7d March 25-31 GSC average position                  No                        No        Allowed; zero can mean no position data
          feat_scroll_7d    March 25-31 GA4 scroll events                  No                        No Borderline: GA4 zero-fill can mean unavailable
            content_type                  Static metadata                  No                        No       Allowed; grouped split guards client mix
observed_forward_decline April totals versus March totals   This is the label                 

### 3b. Error examples

These examples use the existing **After: grouped client split** ranking only; no model is retrained and no new split is created. False positives are the highest-scored rows whose observed April result was not a decline. False negatives are the lowest-scored rows whose observed April result did decline.

The table prints four examples of each error type with the five actual model features, the score, and the observed label. The interpretation below is based on those printed rows, not on an assumed decision threshold.

All eight shown errors are keyword articles, so this small sample does not show a clean content-type separation. The false positives have very low seven-day impressions (1.750–3.400) and zero clicks but score 0.642–0.753, while three false negatives have near-zero impressions and deep positions and one has 413.000 impressions with a 0.009 score. The score ranges do not sit near one borderline threshold: these are observed weak-signal edge cases and, in the 413-impression false negative, a clearly wrong low ranking rather than a small threshold miss.


In [4]:
error_feature_columns = ['feat_impr_7d', 'feat_clicks_7d', 'feat_pos_7d', 'feat_scroll_7d', 'content_type', 'model_score', 'observed_forward_decline']
false_positives = grouped_ranked[grouped_ranked['observed_forward_decline'].eq(0)].head(4).copy()
false_negatives = grouped_ranked[grouped_ranked['observed_forward_decline'].eq(1)].tail(4).sort_values('model_score').copy()
for examples, error_type in [(false_positives, 'false_positive'), (false_negatives, 'false_negative')]:
    examples.insert(0, 'rank_position', examples.index.to_numpy() + 1)
    examples.insert(1, 'error_type', error_type)
error_examples = pd.concat([false_positives, false_negatives], ignore_index=True)
print('Error examples from After: grouped client split (n=4 per error type)')
print(error_examples[['rank_position', 'error_type', *error_feature_columns]].to_string(index=False, formatters={'model_score': '{:.3f}'.format, 'feat_impr_7d': '{:.3f}'.format, 'feat_clicks_7d': '{:.3f}'.format, 'feat_pos_7d': '{:.3f}'.format, 'feat_scroll_7d': '{:.3f}'.format}))
print('\nError-pattern summary')
print(error_examples.groupby(['error_type', 'content_type'], observed=True).size().rename('n').reset_index().to_string(index=False))
print(error_examples.groupby('error_type', observed=True)['model_score'].agg(['min', 'max', 'mean']).to_string(float_format='{:.3f}'.format))

Error examples from After: grouped client split (n=4 per error type)
 rank_position     error_type feat_impr_7d feat_clicks_7d feat_pos_7d feat_scroll_7d    content_type model_score  observed_forward_decline
             1 false_positive        1.750          0.000       5.792          0.750 keyword article       0.753                         0
             4 false_positive        3.400          0.000       7.586          0.400 keyword article       0.680                         0
             5 false_positive        2.000          0.000       4.667          0.333 keyword article       0.673                         0
             9 false_positive        2.167          0.000       2.544          0.167 keyword article       0.642                         0
          2049 false_negative      413.000          8.429       9.361          1.857 keyword article       0.009                         1
          2048 false_negative        1.500          0.000     126.750          0.000 keyword arti

## 4. Claim rewrite

The Week 5 output identified `content_type__content_type_comparison article` as the strongest upward coefficient association in its training split, and it described the model as predicting a probability of decline. Those statements need a careful boundary: a coefficient describes model behavior in this sample; it does not establish a causal lever or guarantee a future result.

The table below shows the original phrasing alongside a narrower rewrite. The code reads the Week 5 notebook as evidence of the original claim and prints both versions.

In [5]:
import nbformat

w05_path = ROOT / 'work' / 'notebooks' / 'w05_model.ipynb'
w05_notebook = nbformat.read(w05_path, as_version=4)
w05_text = '\n'.join(cell.source for cell in w05_notebook.cells)
assert 'Logistic Regression' in w05_text
assert 'strongest upward association with the decline label' in w05_text
claim_rewrites = pd.DataFrame([
    {
        'original claim from Week 5': 'The model predicts a probability that a page will have an observed forward decline.',
        'safe rewrite': 'The model produces a measured score associated with the observed April-versus-March decline label in this warehouse sample; it is a directional decision-support ranking, not a guarantee of future decline.',
    },
    {
        'original claim from Week 5 output': 'content_type__content_type_comparison article is the strongest upward association with the decline label in this training split.',
        'safe rewrite': 'In this measured training split, the comparison-article category received the largest positive coefficient. That describes this fitted model and may reflect category or client mix; it does not show that changing a page into that category causes decline.',
    },
])
print('Read Week 5 source for claim audit:', w05_path.name)
print(claim_rewrites.to_string(index=False))

Read Week 5 source for claim audit: w05_model.ipynb
                                                         original claim from Week 5                                                                                                                                                                                                                                                  safe rewrite                                                                                                original claim from Week 5 output
The model predicts a probability that a page will have an observed forward decline.                                                  The model produces a measured score associated with the observed April-versus-March decline label in this warehouse sample; it is a directional decision-support ranking, not a guarantee of future decline.                                                                                                                              NaN
              

## Self-check

- [x] I named two distinct assigned-paper findings and asked constructive methodology questions tied to their reported pages.
- [x] I re-ran the Week 5 Logistic Regression with random-row and grouped-client validation, showing base rates, client overlap, precision@K, and ROC-AUC for both.
- [x] I audited all five features and the label against timing, client identity, product flags, and known borderline cases.
- [x] I pulled real false-positive and false-negative examples from the grouped-split test set and interpreted the pattern.
- [x] I rewrote Week 5 claims using observed, measured, directional, and decision-support language.
- [x] The notebook runs top to bottom with no errors.
- [x] No client names, URLs, raw private queries, or credentials appear in the notebook.
- [ ] I still need to review the local changes, then commit and push them myself.